# Structured Data RAG: Semantic Search + Text-to-SQL

**Task:** Answer natural language questions over tabular/CSV data using two complementary approaches: semantic row search and text-to-SQL.

| Property | Value |
|----------|-------|
| **Models** | gpt-4o-mini (generation), text-embedding-3-small (embeddings) |
| **Retrieval** | ChromaDB (semantic search) + SQLite (text-to-SQL) |
| **Tools** | pandas, langchain, chromadb, openai, sqlite3 |
| **Skill Level** | Intermediate |
| **Hardware** | CPU (no GPU required) |
| **API Keys Required** | OPENAI_API_KEY |
| **Datasets** | Synthetic product catalog (embedded inline) |

## 1. Imports & Setup

In [ ]:
import os
import json
import sqlite3
from io import StringIO

import pandas as pd
import chromadb
from openai import OpenAI
from langchain_community.utilities import SQLDatabase
from langchain.chains import create_sql_query_chain
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

## 2. Load / Generate Dataset

We'll create a synthetic product catalog. In a real scenario, you'd load your own CSV.

In [ ]:
# Synthetic product catalog
csv_data = '''product_id,product_name,category,price,stock,rating,description
1,Wireless Headphones,Electronics,89.99,45,4.7,Premium noise-cancelling Bluetooth headphones with 30-hour battery life
2,USB-C Cable,Accessories,12.99,200,4.5,Fast charging 6ft USB-C cable with reinforced connectors
3,Laptop Stand,Office,49.99,80,4.6,Adjustable aluminum laptop stand for better ergonomics
4,Mechanical Keyboard,Electronics,129.99,60,4.8,RGB mechanical keyboard with hot-swappable switches
5,Mouse Pad,Accessories,19.99,150,4.4,Extended XL mouse pad with smooth surface
6,Desk Lamp,Office,34.99,110,4.5,LED desk lamp with adjustable color temperature
7,Phone Mount,Accessories,14.99,250,4.3,Universal phone mount for car dashboard
8,Webcam 1080p,Electronics,59.99,90,4.6,Full HD webcam with auto-focus and built-in microphone
9,External SSD 1TB,Electronics,99.99,70,4.7,Portable 1TB SSD with 550MB/s read speed
10,Screen Protector,Accessories,8.99,300,4.2,Tempered glass screen protector for phones'''

df = pd.read_csv(StringIO(csv_data))
print(f'Loaded {len(df)} products:')
df.head()

## 3. Approach 1: Semantic Row Search with ChromaDB

Serialize each row as a natural-language string, embed it, store in ChromaDB, then retrieve and answer.

In [ ]:
def serialize_row(row):
    """Convert a DataFrame row to a natural language string."""
    return (
        f"Product: {row['product_name']}. "
        f"Category: {row['category']}. "
        f"Price: ${row['price']}. "
        f"Stock: {row['stock']} units. "
        f"Rating: {row['rating']}/5. "
        f"Description: {row['description']}"
    )

# Create ChromaDB client and collection
chroma_client = chromadb.EphemeralClient()
collection = chroma_client.create_collection(name='products')

# Serialize and embed each row
for idx, row in df.iterrows():
    text = serialize_row(row)
    collection.add(
        ids=[str(row['product_id'])],
        documents=[text],
        metadatas=[{"product_name": row['product_name'], "price": row['price']}]
    )

print(f'Indexed {len(df)} products in ChromaDB')

In [ ]:
def semantic_search(question: str, k: int = 3) -> str:
    """Retrieve relevant products and generate an answer via LLM."""
    # Query ChromaDB
    results = collection.query(
        query_texts=[question],
        n_results=k
    )
    
    context = '\n'.join(results['documents'][0])
    
    # Generate answer with LLM
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {
                'role': 'system',
                'content': 'You are a helpful product assistant. Answer questions based on the product information provided.'
            },
            {
                'role': 'user',
                'content': f'Product Information:\n{context}\n\nQuestion: {question}'
            }
        ],
        temperature=0
    )
    
    return response.choices[0].message.content

# Test semantic search
test_question = 'What are the best noise-cancelling audio products?'
answer = semantic_search(test_question)
print(f'Q: {test_question}')
print(f'A: {answer}')

## 4. Approach 2: Text-to-SQL with LangChain

Convert natural language to SQL, execute against SQLite, and return structured results.

In [ ]:
# Create in-memory SQLite database
import sqlalchemy
engine = sqlalchemy.create_engine('sqlite:///:memory:')
df.to_sql('products', engine, index=False, if_exists='replace')
db = SQLDatabase(engine)

print('SQLite database created with products table')
print(f'Table schema: {db.get_table_info()}')

In [ ]:
# Create SQL query chain
chain = create_sql_query_chain(llm, db)

def text_to_sql(question: str) -> str:
    """Convert natural language question to SQL and execute."""
    response = chain.invoke({"input": question})
    return response

# Test text-to-SQL
test_question = 'What are the top 3 highest-rated products?'
answer = text_to_sql(test_question)
print(f'Q: {test_question}')
print(f'A: {answer}')

## 5. Side-by-Side Comparison

Run the same questions through both approaches.

In [ ]:
test_questions = [
    'What products are under $30?',
    'Which electronics have the highest rating?',
    'Show me office products with good stock levels'
]

results = []
for q in test_questions:
    sem_ans = semantic_search(q, k=2)
    sql_ans = text_to_sql(q)
    results.append({
        'Question': q,
        'Semantic Search': sem_ans[:150] + '...' if len(sem_ans) > 150 else sem_ans,
        'Text-to-SQL': sql_ans[:150] + '...' if len(sql_ans) > 150 else sql_ans
    })

comparison_df = pd.DataFrame(results)
print(comparison_df.to_string(index=False))

## When to Use Each Approach

| Approach | Best For | Pros | Cons |
|----------|----------|------|------|
| **Semantic Search** | Exploratory queries, fuzzy matching, narrative descriptions | Handles nuanced language, fast, low latency | Less precise for numerical aggregations |
| **Text-to-SQL** | Aggregations, exact filters, complex joins | Precise results, scales to large datasets | Requires good schema documentation, fails on ambiguous queries |